# One-token categorical sampler example

This notebook uses a single categorical token with a known target distribution.
The exact posterior \(p(x_0 \mid x_t)\) is available analytically because the
clean token is represented by a one-hot vector and the forward process is an
Ornstein--Uhlenbeck Gaussian corruption.

The experiment compares:

- `ode`: probability-flow ODE Euler steps using the exact posterior mean.
- `sde`: reverse SDE Euler steps using the exact score and Gaussian noise.
- `marginally_exact`: sample the categorical endpoint from the exact posterior,
  then take the exact OU bridge transition to the next time.


In [ ]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

VOCAB_SIZE = 6
T = 4.0
NUM_SAMPLES = 200_000
STEPS_TO_TEST = [1, 2, 4, 8, 16, 32, 64]
SAMPLERS = ["ode", "sde", "marginally_exact"]

print("device:", DEVICE)


In [ ]:
target_probs = torch.tensor(
    [0.38, 0.24, 0.15, 0.10, 0.08, 0.05],
    dtype=DTYPE,
    device=DEVICE,
)
target_probs = target_probs / target_probs.sum()
basis = torch.eye(VOCAB_SIZE, dtype=DTYPE, device=DEVICE)


def total_variation(p, q):
    return 0.5 * (p - q).abs().sum().item()


print("target:", target_probs.detach().cpu().tolist())


In [ ]:
def ou_c(u):
    u = torch.as_tensor(u, dtype=DTYPE, device=DEVICE)
    return torch.exp(-u)


def ou_sigma2(u):
    u = torch.as_tensor(u, dtype=DTYPE, device=DEVICE)
    return 1.0 - torch.exp(-2.0 * u)


def exact_posterior_probs(y, u):
    """Return p(x0 = e_i | x_u = y) for one categorical token."""
    c_u = ou_c(u)
    s2_u = ou_sigma2(u).clamp_min(1e-14)
    means = c_u * basis
    diff = y[:, None, :] - means[None, :, :]
    log_likelihood = -0.5 * diff.square().sum(dim=-1) / s2_u
    logits = log_likelihood + target_probs.log()[None, :]
    return torch.softmax(logits, dim=-1)


def posterior_mean(y, u):
    probs = exact_posterior_probs(y, u)
    return probs @ basis


def exact_score(y, u):
    c_u = ou_c(u)
    s2_u = ou_sigma2(u).clamp_min(1e-14)
    return (c_u * posterior_mean(y, u) - y) / s2_u


In [ ]:
def apply_temperature_and_nucleus(probs, temperature=1.0, top_p=1.0):
    """Optional decoding controls for the endpoint categorical draw."""
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    adjusted = probs
    if temperature != 1.0:
        adjusted = torch.softmax(adjusted.clamp_min(1e-300).log() / temperature, dim=-1)

    if top_p < 1.0:
        sorted_probs, sorted_idx = adjusted.sort(dim=-1, descending=True)
        cdf = sorted_probs.cumsum(dim=-1)
        keep = cdf <= top_p
        keep[..., 0] = True
        sorted_filtered = torch.where(keep, sorted_probs, torch.zeros_like(sorted_probs))
        filtered = torch.zeros_like(adjusted).scatter(dim=-1, index=sorted_idx, src=sorted_filtered)
        adjusted = filtered / filtered.sum(dim=-1, keepdim=True).clamp_min(1e-300)

    return adjusted


def sample_endpoint(probs, temperature=1.0, top_p=1.0):
    probs = apply_temperature_and_nucleus(probs, temperature=temperature, top_p=top_p)
    ids = torch.multinomial(probs.to(torch.float32), num_samples=1).squeeze(-1)
    return F.one_hot(ids, VOCAB_SIZE).to(dtype=DTYPE, device=DEVICE), ids


In [ ]:
def ou_bridge_step(y, u, du, endpoint, add_noise=True):
    """Exact OU bridge transition from forward time u to max(u - du, 0)."""
    u_next = max(float(u) - float(du), 0.0)
    if u_next <= 1e-12:
        return endpoint

    c_u = ou_c(u)
    c_next = ou_c(u_next)
    c_du = ou_c(du)
    s2_u = ou_sigma2(u).clamp_min(1e-14)
    s2_next = ou_sigma2(u_next).clamp_min(1e-14)

    gain = c_du * s2_next / s2_u
    mean = c_next * endpoint + gain * (y - c_u * endpoint)
    var = (s2_next - (c_du.square() * s2_next.square() / s2_u)).clamp_min(0.0)

    if not add_noise:
        return mean
    return mean + var.sqrt() * torch.randn_like(mean)


In [ ]:
def ode_step(y, u, du):
    score = exact_score(y, u)
    drift = y + score
    return y + du * drift


def sde_step(y, u, du):
    score = exact_score(y, u)
    drift = y + 2.0 * score
    return y + du * drift + math.sqrt(2.0 * du) * torch.randn_like(y)


def marginally_exact_step(y, u, du, temperature=1.0, top_p=1.0):
    probs = exact_posterior_probs(y, u)
    endpoint, _ = sample_endpoint(probs, temperature=temperature, top_p=top_p)
    return ou_bridge_step(y, u, du, endpoint, add_noise=True)


In [ ]:
def decode(y):
    return y.argmax(dim=-1)


def empirical_distribution(ids):
    counts = torch.bincount(ids, minlength=VOCAB_SIZE).to(DTYPE)
    return counts / counts.sum()


@torch.no_grad()
def run_sampler(name, steps, num_samples=NUM_SAMPLES, temperature=1.0, top_p=1.0):
    y = torch.randn(num_samples, VOCAB_SIZE, dtype=DTYPE, device=DEVICE)
    u = T
    du = T / steps

    for _ in range(steps):
        if name == "ode":
            y = ode_step(y, u, du)
        elif name == "sde":
            y = sde_step(y, u, du)
        elif name == "marginally_exact":
            y = marginally_exact_step(y, u, du, temperature=temperature, top_p=top_p)
        else:
            raise ValueError(name)
        u = max(u - du, 0.0)

    ids = decode(y)
    return empirical_distribution(ids)


In [ ]:
rows = []
for steps in STEPS_TO_TEST:
    for sampler in SAMPLERS:
        empirical = run_sampler(sampler, steps)
        rows.append(
            {
                "sampler": sampler,
                "steps": steps,
                "tv": total_variation(empirical, target_probs),
                "empirical": empirical.detach().cpu().tolist(),
            }
        )

for row in rows:
    print(f"{row['sampler']:18s} steps={row['steps']:3d} tv={row['tv']:.5f}")


In [ ]:
# Optional plot. This cell is safe to skip on machines without matplotlib.
try:
    import matplotlib.pyplot as plt

    for sampler in SAMPLERS:
        xs = [row["steps"] for row in rows if row["sampler"] == sampler]
        ys = [row["tv"] for row in rows if row["sampler"] == sampler]
        plt.plot(xs, ys, marker="o", label=sampler)
    plt.xscale("log", base=2)
    plt.xlabel("number of reverse steps")
    plt.ylabel("TV distance to target categorical distribution")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
except Exception as exc:
    print("Plot skipped:", exc)
